# Chebyshev Sine Recurrence

This notebook documents the efficient sine generation technique used inside
`source/synth/Oscillator.cpp` (and `SineOscillator.cpp`).

Synthesisers need `sin(phase)` evaluated on every audio sample — potentially
millions of times per second across multiple voices. Calling `std::sin()`
each time is expensive because it uses a software approximation (CORDIC or
polynomial) that costs tens of CPU cycles per call.

The Chebyshev recurrence reduces per-sample cost to **one multiply and one
subtract**, at the price of accumulated numerical error that must be managed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

PI = np.pi

## 1. The addition formula

The sine addition formula gives us:

$$\sin(\theta + \Delta) = \sin(\theta)\cos(\Delta) + \cos(\theta)\sin(\Delta)$$

At each step we advance the phase by the constant $\Delta$ (the phase
increment per sample). If we track both $\sin(\theta)$ and $\cos(\theta)$
we can compute the next sample in two multiplies and two adds:

$$\sin_{n+1} = \sin_n \cos(\Delta) + \cos_n \sin(\Delta)$$
$$\cos_{n+1} = \cos_n \cos(\Delta) - \sin_n \sin(\Delta)$$

But this requires maintaining **both** $\sin$ and $\cos$ state — four multiplies
per sample total.

## 2. The Chebyshev identity

Writing the same formula for $n-1$:

$$\sin((n-1)\Delta) = \sin(n\Delta)\cos(\Delta) - \cos(n\Delta)\sin(\Delta)$$

Eliminate $\cos(n\Delta)$ by adding to the $n+1$ equation:

$$\boxed{\sin((n+1)\Delta) = 2\cos(\Delta) \cdot \sin(n\Delta) - \sin((n-1)\Delta)}$$

This is the **Chebyshev recurrence**. Given the two previous sine values,
the next one costs only **one multiply** (by the constant $2\cos(\Delta)$)
and **one subtract** — no cosine state needed.

In the Synple code:
```cpp
// sinRecurrenceCoeff_ = 2 * cos(increment_)
const float sinp = sinRecurrenceCoeff_ * sinN_ - sinNm1_;
sinNm1_ = sinN_;
sinN_ = sinp;
```

In [ ]:
def chebyshev_sine(delta, n_samples, amplitude=1.0, phase0=0.0):
    """
    Generate sine values using the Chebyshev recurrence.
    delta: phase increment per sample (radians)
    Returns array of sin(phase0 + n*delta) for n = 0..n_samples-1.
    """
    coeff = 2.0 * np.cos(delta)           # sinRecurrenceCoeff_
    sin_n   = amplitude * np.sin(phase0)  # sinN_
    sin_nm1 = amplitude * np.sin(phase0 - delta)  # sinNm1_

    out = [sin_n]
    for _ in range(n_samples - 1):
        sinp = coeff * sin_n - sin_nm1
        sin_nm1 = sin_n
        sin_n = sinp
        out.append(sin_n)
    return np.array(out)


# Demonstrate correctness for a few cycles
fs = 44100
f0 = 440.0
delta = 2 * PI * f0 / fs
N = 400

recurrence = chebyshev_sine(delta, N)
reference  = np.sin(np.arange(N) * delta)  # std::sin reference

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

t = np.arange(N) / fs * 1000
axes[0].plot(t, recurrence, color='seagreen', lw=2, label='Chebyshev recurrence')
axes[0].plot(t, reference,  color='steelblue', lw=1.5, ls='--', alpha=0.7, label='std::sin reference')
axes[0].set_ylabel('Amplitude')
axes[0].set_title(f'440 Hz sine — first {N} samples ({N/fs*1000:.1f} ms)')
axes[0].legend()

error = recurrence - reference
axes[1].plot(t, error, color='tomato', lw=1)
axes[1].set_ylabel('Error (recurrence − reference)')
axes[1].set_xlabel('Time (ms)')
axes[1].set_title('Error is near float epsilon initially — grows over many cycles')

plt.tight_layout()
plt.show()

print(f"Max error over {N} samples: {np.max(np.abs(error)):.2e}")

## 3. Initialisation

The recurrence requires two initial values:

```cpp
sinN_   = amplitude_ * std::sin(phase_);
sinNm1_ = amplitude_ * std::sin(phase_ - increment_);
sinRecurrenceCoeff_ = 2.0f * std::cos(increment_);
```

These are computed once at the start of each half-period (inside the
`phase_ ≤ quarterPi` branch of `nextSample()`). The two `std::sin` calls
are the only transcendental functions executed per half-period — not per
sample.

At a typical audio frequency of 440 Hz and sample rate 44 100 Hz, one
half-period is ~50 samples. So the initialization overhead is about 2
transcendental calls per 50 samples — effectively free compared to calling
`std::sin` on every sample.

In [ ]:
# How often does initialization occur?
test_freqs = [55, 110, 220, 440, 880, 1760, 3520, 7040]
print("Initialization frequency:")
print(f"  {'Freq':>6}  {'Half-period (samples)':>22}  {'Init calls / sec':>17}")
print("  " + "-" * 50)
for f in test_freqs:
    half_period = fs / f / 2
    inits_per_sec = f * 2  # two half-periods per cycle
    print(f"  {f:>4} Hz  {half_period:>22.1f}  {inits_per_sec:>15} /s")

## 4. Numerical stability — phase drift over time

Floating-point arithmetic is not exact. Each step of the recurrence
introduces a tiny rounding error. These errors accumulate over many cycles,
causing the computed sine to drift away from the true value.

The drift is not random: the recurrence corresponds to a second-order IIR
filter with poles exactly on the unit circle. Numerical errors perturb those
poles slightly, causing the output to grow (or shrink) exponentially over
very long runs.

The Synple oscillator sidesteps this entirely: the recurrence is only run
for **one half-period** (typically 20–100 samples) before it is
re-initialised from `std::sin`. This bounds the accumulated error to a
single half-period's worth, which is negligible.

In [ ]:
# Show error accumulation over many cycles WITHOUT periodic reinitialisation
N_long = 200_000   # ~4.5 seconds at 44.1 kHz
f0_test = 440.0
delta = 2 * PI * f0_test / fs

recurrence_long = chebyshev_sine(delta, N_long, phase0=0.0)
reference_long  = np.sin(np.arange(N_long, dtype=np.float32) * np.float32(delta))

error_long = np.abs(recurrence_long - reference_long.astype(float))

# With periodic reinitialisation every half-period
half_period_samples = int(round(fs / f0_test / 2))

def chebyshev_with_reinit(delta, n_total, reinit_interval):
    coeff = 2.0 * np.cos(delta)
    out = []
    for start in range(0, n_total, reinit_interval):
        phase0 = start * delta
        sin_n   = np.sin(phase0)
        sin_nm1 = np.sin(phase0 - delta)
        chunk_len = min(reinit_interval, n_total - start)
        out.append(sin_n)
        for _ in range(chunk_len - 1):
            sinp = coeff * sin_n - sin_nm1
            sin_nm1 = sin_n
            sin_n = sinp
            out.append(sin_n)
    return np.array(out)

recurrence_reinit = chebyshev_with_reinit(delta, N_long, half_period_samples)
error_reinit = np.abs(recurrence_reinit - reference_long.astype(float))

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

t_long = np.arange(N_long) / fs
axes[0].semilogy(t_long, error_long + 1e-40, color='tomato', lw=0.8,
                 label='No reinitialisation')
axes[0].semilogy(t_long, error_reinit + 1e-40, color='seagreen', lw=0.8,
                 label=f'Reinit every {half_period_samples} samples (Synple)')
axes[0].set_ylabel('|error|')
axes[0].set_title('Phase error accumulation over time (440 Hz, 44.1 kHz)')
axes[0].legend()

# Zoom into first 0.5 seconds
mask = t_long < 0.5
axes[1].semilogy(t_long[mask], error_long[mask] + 1e-40, color='tomato', lw=1,
                 label='No reinit — error grows')
axes[1].semilogy(t_long[mask], error_reinit[mask] + 1e-40, color='seagreen', lw=1,
                 label='With reinit — bounded')
axes[1].set_ylabel('|error|')
axes[1].set_xlabel('Time (s)')
axes[1].set_title('Zoomed: first 0.5 s')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"Without reinit — max error after {N_long/fs:.1f}s: {np.max(error_long):.2e}")
print(f"With reinit    — max error after {N_long/fs:.1f}s: {np.max(error_reinit):.2e}")

## 5. Why the BLEP oscillator reinitialises every half-period

The BLEP oscillator's phase lifecycle naturally triggers re-initialisation
every half-period: when `phase_ ≤ quarterPi`, the oscillator is at (or
crossing) zero in phase space. At that moment it recomputes:

```cpp
sinN_   = amplitude_ * std::sin(phase_);
sinNm1_ = amplitude_ * std::sin(phase_ - increment_);
sinRecurrenceCoeff_ = 2.0f * std::cos(increment_);
```

This is not just about numerical stability — it is also how frequency
changes (`setPeriod()`) take effect cleanly without clicks, since
`increment_` and `halfPhase_` are recomputed from the current `period_`
at each half-period boundary.

Two benefits in one:
1. **Stability**: accumulated error is flushed every ~20–100 samples
2. **Smoothness**: frequency updates take effect at a zero-crossing

In [ ]:
# Show error per half-period chunk — it resets to float-epsilon each time
freqs_test = [110, 440, 2000, 7000]
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(freqs_test)))

fig, ax = plt.subplots(figsize=(12, 4))

for f, color in zip(freqs_test, colors):
    d = 2 * PI * f / fs
    hp = int(round(fs / f / 2))
    N_show = hp * 20   # 20 half-periods

    rec = chebyshev_with_reinit(d, N_show, hp)
    ref = np.sin(np.arange(N_show) * d)
    err = np.abs(rec - ref)

    t_ms = np.arange(N_show) / fs * 1000
    ax.semilogy(t_ms, err + 1e-40, color=color, lw=1.2, label=f'{f} Hz (hp={hp} samples)')

    # Mark reinit points
    for k in range(20):
        ax.axvline(k * hp / fs * 1000, color=color, alpha=0.15, lw=0.6)

ax.set_xlabel('Time (ms)')
ax.set_ylabel('|error|')
ax.set_title('Error per half-period — resets to ~float epsilon at each reinitialisation')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Computational cost comparison

| Method | Operations per sample | Transcendental calls |
|--------|-----------------------|---------------------|
| `std::sin()` direct | ~20–40 (polynomial approx) | 1 per sample |
| Addition formula (sin+cos) | 4 multiplies, 2 adds | 0 per sample (after init) |
| **Chebyshev recurrence** | **1 multiply, 1 subtract** | **0 per sample (after init)** |

The two `std::sin` initialisation calls happen every half-period
(~ once per 20–100 samples), so the amortised cost is:

In [ ]:
# Amortised transcendental call cost vs frequency
freqs = np.linspace(20, 20000, 500)
half_periods = fs / freqs / 2
# 2 std::sin calls per half-period, amortised across half_period samples
amortised_sin_calls = 2.0 / half_periods

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(freqs, amortised_sin_calls, color='darkorchid', lw=2)
ax.axhline(1.0, color='tomato', ls='--', lw=1.5, label='std::sin per sample (direct)')
ax.set_xlabel('Oscillator frequency (Hz)')
ax.set_ylabel('Amortised std::sin calls per sample')
ax.set_title('Chebyshev recurrence: amortised transcendental cost vs frequency')
ax.set_yscale('log')
ax.legend()
plt.tight_layout()
plt.show()

print("Amortised std::sin calls per sample:")
for f in [110, 440, 1000, 4000, 10000]:
    hp = fs / f / 2
    cost = 2 / hp
    print(f"  {f:>6} Hz  (half-period = {hp:.1f} samples)  →  {cost:.4f} calls/sample")

## 7. Summary

| Property | Detail |
|----------|--------|
| **Recurrence** | `sinN = 2·cos(Δ)·sinNm1 − sinNm2` |
| **Coefficient** | `sinRecurrenceCoeff_ = 2·cos(increment_)` — computed once per half-period |
| **Init** | Two `std::sin` calls at the start of each half-period |
| **Per-sample cost** | 1 multiply + 1 subtract (vs ~20–40 ops for `std::sin`) |
| **Stability** | Error bounded to one half-period by periodic reinitialisation |
| **Frequency updates** | Parameter changes take effect at the next half-period boundary (no clicks) |
| **Used in** | `Oscillator::nextSample()` — the main synth oscillator |
| **Also in** | `SineOscillator::nextSample()` — LFO-style sine generator |

The combination of Chebyshev recurrence + half-period reinitialisation gives
near-`std::sin` accuracy at a fraction of the computational cost, which is
essential for running 8+ voice polyphony in real time.